# FollowSM x Polymarket: Cross-Venue Risk Quickstart

This notebook pulls a live **Binance x Polymarket confluence snapshot** from FollowSM, re-derives the HFT risk ladder with your own `RiskConfig`, and then runs the starter-kit bot in **paper mode** for one minute.

No API key is needed: requests without a key run on the free, IP-rate-limited tier. Colab shares egress IPs, so if you see a rate-limit message, grab a key at https://follow-sm.com/pricing.

In [ ]:
!pip install -q "followsm-sdk>=1.2.0"

In [ ]:
import getpass

from followsm_sdk import FollowSMClient, RateLimitExceededException, RiskConfig

API_KEY = getpass.getpass("FollowSM API key (press Enter for the free tier): ").strip() or None

risk_config = RiskConfig(vpin_percentile_widen_threshold=0.85, vpin_percentile_halt_threshold=0.97, min_semantic_confidence=0.70)
client = FollowSMClient(api_key=API_KEY, risk_config=risk_config)

In [ ]:
try:
    snapshot = client.get_confluence_snapshot("BTCUSDT")
except RateLimitExceededException as exc:
    print(exc)
    print("Upgrade to DEVELOPER ($199/mo) or ENTERPRISE ($499/mo): https://follow-sm.com/pricing")
    raise

micro = snapshot.binance_microstructure
print(f"BTCUSDT  price={micro.price:,.2f}  vpin={micro.vpin:.3f}  vpin_percentile={micro.vpin_percentile}  ob_toxicity_1pct={micro.ob_toxicity_1pct:.2f}")
print(f"spot move 15m = {micro.price_delta_15m_pct * 100:+.3f}%")
for event in snapshot.polymarket_confluence.active_events:
    print(f"  {event.market_slug:<55} p={event.implied_probability:.3f}  prob_delta_15m={event.prob_delta_15m:+.3f}  dir={event.direction}")

In [ ]:
print("Backend recommended_action :", snapshot.composite_signals.recommended_action)
print("Your RiskConfig ladder     :", client.evaluate_risk(snapshot))
print("Cross-market divergence    :", snapshot.composite_signals.cross_market_divergence_flag)

## Run the bot in paper mode (60 seconds)

Paper mode reads real Polymarket CLOB midpoints but never sends orders.

In [ ]:
!git clone -q https://github.com/Follow-SM/polymarket-arbitrage-starter-kit.git
%cd polymarket-arbitrage-starter-kit
!pip install -q -r requirements.txt
!cp .env.example .env

In [ ]:
import os

if API_KEY:
    os.environ["FOLLOWSM_API_KEY"] = API_KEY
!timeout -s INT 60 python -m bot